# Forecast-to-Inventory Decision Simulator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmavigo/retail-demand-forecasting/blob/main/notebooks/04_inventory_decisions.ipynb)

This notebook starts from raw Kaggle demand, chooses a forecast on an unseen horizon and translates it into safety stock, reorder points, stockouts and excess units.

## 1. Environment and official data

In [ ]:
import sys, subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'kagglehub>=1.0,<2', 'pandas>=2.2,<3', 'numpy>=2,<3',
        'plotly>=5.24,<7', 'scikit-learn>=1.5,<2', 'lightgbm>=4.5,<5'
    ])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
pd.set_option('display.max_columns', 30)


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('m5-forecasting-accuracy')

print("Path to competition files:", path)

DATA_DIR = Path(path)
if not (DATA_DIR / 'calendar.csv').exists():
    DATA_DIR = next(p.parent for p in DATA_DIR.rglob('calendar.csv'))

required = {'calendar.csv', 'sell_prices.csv', 'sales_train_evaluation.csv'}
available = {p.name for p in DATA_DIR.glob('*.csv')}
assert required.issubset(available), f"Missing files: {sorted(required - available)}"


## 2. Load demand and create operational forecast candidates

In [ ]:
sales = pd.read_csv(DATA_DIR / 'sales_train_evaluation.csv')
calendar = pd.read_csv(DATA_DIR / 'calendar.csv', parse_dates=['date'])
day_cols = [c for c in sales if c.startswith('d_')]
values = sales[day_cols].to_numpy(dtype=np.float32)
HORIZON = 28
TRAIN_END = values.shape[1] - HORIZON
train, actual = values[:, :TRAIN_END], values[:, TRAIN_END:]

mean_28 = np.repeat(train[:, -28:].mean(axis=1, keepdims=True), HORIZON, axis=1)
lag_28 = train[:, -28:].copy()
lag_7 = np.tile(train[:, -7:], (1, 4))
forecast_candidates = {
    'Mean of last 28 days': mean_28,
    'Seasonal lag 28': lag_28,
    'Seasonal lag 7': lag_7,
    '75% mean + 25% seasonal': .75 * mean_28 + .25 * lag_28,
}


In [ ]:
def score(actual, predicted, history):
    error = actual - predicted
    scale = np.mean(np.diff(history, axis=1) ** 2, axis=1)
    usable = scale > 0
    denominator = np.abs(actual).sum()
    return {
        'MAE': float(np.abs(error).mean()),
        'WAPE': float(np.abs(error).sum() / denominator),
        'RMSSE': float(np.sqrt(np.mean(error[usable] ** 2, axis=1) / scale[usable]).mean()),
        'Bias': float(error.sum() / denominator),
    }


In [ ]:
metrics = pd.DataFrame([{'method': n, **score(actual, p, train)} for n, p in forecast_candidates.items()]).sort_values('WAPE')
display(metrics.style.format({'MAE': '{:.4f}', 'WAPE': '{:.2%}', 'RMSSE': '{:.4f}', 'Bias': '{:.2%}'}))
best_method = metrics.iloc[0].method
forecast = forecast_candidates[best_method]


## 3. Define inventory assumptions
The example uses a seven-day lead time and a 95% target service level (`z = 1.65`). These are editable business assumptions, not facts contained in M5.

In [ ]:
LEAD_TIME_DAYS = 7
SERVICE_Z = 1.65
UNIT_HOLDING_COST = 1.0
UNIT_STOCKOUT_COST = 3.0

residual = train[:, -84:] - np.repeat(train[:, -112:-84].mean(axis=1, keepdims=True), 84, axis=1)
residual_std = residual.std(axis=1)
safety_stock = SERVICE_Z * residual_std * np.sqrt(LEAD_TIME_DAYS)
demand_during_lead_time = forecast[:, :LEAD_TIME_DAYS].sum(axis=1)


## 4. Calculate decisions for every product-store series

In [ ]:
inventory = sales[['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']].copy()
inventory['actual_units'] = actual.sum(axis=1)
inventory['forecast_units'] = forecast.sum(axis=1)
inventory['safety_stock'] = safety_stock
inventory['reorder_point'] = demand_during_lead_time + safety_stock
inventory['stockout_units'] = (inventory.actual_units - inventory.forecast_units).clip(lower=0)
inventory['excess_units'] = (inventory.forecast_units - inventory.actual_units).clip(lower=0)
inventory['estimated_cost'] = inventory.stockout_units * UNIT_STOCKOUT_COST + inventory.excess_units * UNIT_HOLDING_COST
display(inventory.head(10))


## 5. Compare operating scenarios

In [ ]:
scenario_rows = []
for method, prediction in forecast_candidates.items():
    order = prediction.sum(axis=1)
    actual_total = actual.sum(axis=1)
    stockout = np.maximum(actual_total - order, 0)
    excess = np.maximum(order - actual_total, 0)
    scenario_rows.append({
        'method': method,
        'stockout_units': stockout.sum(),
        'excess_units': excess.sum(),
        'estimated_cost': stockout.sum() * UNIT_STOCKOUT_COST + excess.sum() * UNIT_HOLDING_COST,
        'service_level': 1 - (stockout > 0).mean(),
    })
scenarios = pd.DataFrame(scenario_rows).sort_values('estimated_cost')
display(scenarios.style.format({'stockout_units': '{:,.0f}', 'excess_units': '{:,.0f}', 'estimated_cost': '${:,.0f}', 'service_level': '{:.1%}'}))
px.bar(scenarios.melt('method', value_vars=['stockout_units', 'excess_units'], var_name='outcome', value_name='units'), x='method', y='units', color='outcome', barmode='group', title='Inventory trade-off by forecast method').show()


## 6. Prioritize products for action

In [ ]:
priority = inventory.sort_values(['stockout_units', 'estimated_cost'], ascending=False)
display(priority[['store_id', 'item_id', 'cat_id', 'forecast_units', 'safety_stock', 'reorder_point', 'stockout_units', 'excess_units', 'estimated_cost']].head(25))
px.bar(priority.head(20).sort_values('estimated_cost'), x='estimated_cost', y='item_id', color='store_id', orientation='h', title='Highest-cost product-store risks').show()


## Limitations
M5 does not contain inventory on hand, supplier lead times, purchase orders or lost-sales records. The simulator demonstrates the decision framework; production use requires those operational inputs and business-approved cost assumptions.